# Bronze → Silver | ACDOCA e Dimensões SAP (com Data Quality)

Notebook PySpark (Fabric) que lê as tabelas da camada **Bronze**, seleciona apenas as colunas relevantes para análise, aplica checagens de **Data Quality** (nulos e duplicidade) e grava o resultado na camada **Silver**.

**Sobre o nome das colunas na bronze:** cada coluna tem, literalmente, o texto `d.results.<campo>` como nome (ex: `d.results.CompanyCode`), resquício do processo de ingestão que achatou o JSON do OData usando o path inteiro como string — **não é uma struct aninhada**. Por isso, toda referência a essas colunas no código usa crase (`` `d.results.CompanyCode` ``): sem a crase, o Spark interpretaria o nome como acesso a campo aninhado (`d` → `results` → `CompanyCode`), que não existe, e o notebook quebraria.

Abordagem orientada a configuração: um dicionário descreve, por entidade, quais colunas manter (com o novo nome de negócio) e qual é a chave de negócio usada para checar duplicidade.

In [1]:
# Bibliotecas do PySpark utilizadas no notebook
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from datetime import datetime

# No Fabric a sessão Spark já vem criada, mas o getOrCreate() garante
# que o notebook também rode fora do ambiente gerenciado, se precisar
spark = SparkSession.builder.getOrCreate()


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 3, Finished, Available, Finished, False)

In [2]:
# ---------------------------------------------------------------
# PARÂMETROS DO NOTEBOOK
# Troque aqui para reaproveitar o notebook em outros ambientes/lakehouses
# (ex: rodar contra lh_sap_silver -> lh_sap_gold no futuro)
# ---------------------------------------------------------------
LAKEHOUSE_BRONZE = "lh_sap_bronze"   # lakehouse de origem (dados brutos)
LAKEHOUSE_SILVER = "lh_sap_silver"   # lakehouse de destino (dados tratados)
SCHEMA_BRONZE = "dbo"                # schema das tabelas na bronze
SCHEMA_SILVER = "dbo"                # schema das tabelas na silver

# Prefixo literal presente em TODAS as colunas da bronze (confirmado no
# schema exportado) — usado para montar o nome real da coluna na hora do select
PREFIXO_COLUNA_BRONZE = "d.results."


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 4, Finished, Available, Finished, False)

## 1. Configuração das entidades

Para cada tabela bronze definimos:
- **colunas**: quais colunas trazer da API e qual nome de negócio (em português) elas recebem na silver — aqui usamos só o nome "limpo" do campo (ex: `CompanyCode`), sem o prefixo; o prefixo é adicionado automaticamente na hora de montar o select
- **chave_negocio**: coluna(s) que identificam um registro de forma única — usada na checagem de duplicidade

Manter isso em um dicionário (em vez de repetir o código 8 vezes) facilita adicionar uma nova entidade ou ajustar colunas sem duplicar lógica.

In [3]:
CONFIG_ENTIDADES = {

   # Dimensão: Centro de Custo (CostCenter)
    "centro_custo": {
        "colunas": {
            "CostCenter": "centro_custo",
            "ControllingArea": "area_controladoria",
            "CostCenterName": "nome_centro_custo",
            "Language": "idioma",
        },
        "chave_negocio": ["centro_custo"],
    },

    # Dimensão: Centro de Lucro (ProfitCenter)
    "centro_lucro": {
        "colunas": {
            "ProfitCenter": "centro_lucro",
            "ControllingArea": "area_controladoria",
            "ProfitCenterName": "nome_centro_lucro",
            "Language": "idioma",
        },
        "chave_negocio": ["centro_lucro"],
    },

    # Dimensão: Cliente/Fornecedor (Business Partner)
    # Customer e Supplier são "papéis" que o mesmo parceiro pode assumir
    "cliente_fornecedor": {
        "colunas": {
            "BusinessPartner": "parceiro_negocio",
            "BusinessPartnerCategory": "categoria_parceiro",
            "BusinessPartnerFullName": "nome_completo",
            "FirstName": "primeiro_nome",
            "LastName": "ultimo_nome",
            "Customer": "cliente",
            "Supplier": "fornecedor",
            "BusinessPartnerIsBlocked": "parceiro_bloqueado",
            "CreationDate": "data_criacao",
        },
        "chave_negocio": ["parceiro_negocio"],
    },

    # Dimensão: Empresa (Company Code)
    "empresa": {
        "colunas": {
            "CompanyCode": "empresa",
            "CompanyCodeName": "nome_empresa",
            "Country": "pais",
            "Currency": "moeda",
            "ChartOfAccounts": "plano_contas",
            "ControllingArea": "area_controladoria",
            "FiscalYearVariant": "variante_ano_fiscal",
            "Language": "idioma",
        },
        "chave_negocio": ["empresa"],
    },

    # Dimensão: Conta Contábil (GL Account)
    # A chave é composta pois a mesma conta pode existir em mais de um plano de contas
    "conta_contabil": {
        "colunas": {
            "ChartOfAccounts": "plano_contas",
            "GLAccount": "conta_contabil",
            "GLAccountName":"nome_conta_contabil",
            "Language": "idioma",
        },
        "chave_negocio": ["plano_contas", "conta_contabil"],
    },

    # Dimensão: Segmento (apenas o código; sem texto descritivo disponível na bronze)
    "segmento": {
        "colunas": {
            "Segment": "segmento",
        },
        "chave_negocio": ["segmento"],
    },

    # Fato: Lançamentos de Despesas (ACDOCA)
    # Grão: um item de lançamento contábil por linha
    "lancamento_despesas": {
        "colunas": {
            "ID": "id_lancamento",
            "CompanyCode": "empresa",
            "ControllingArea": "area_controladoria",
            "FiscalPeriod": "periodo_fiscal",
            "FiscalYearPeriod": "ano_periodo_fiscal",
            "LedgerFiscalYear": "ano_fiscal",
            "Ledger": "ledger",
            "GLAccount": "conta_contabil",
            "CostCenter": "centro_custo",
            "ProfitCenter": "centro_lucro",
            "Segment": "segmento",
            "Customer": "cliente",
            #"PartnerCompany": "cliente_fornecedor",
            #"PartnerCompanyCode": "cod_cliente_fornecedor",
            "BusinessTransactionType": "tipo_transacao",
            "AmountInCompanyCodeCurrency": "valor",
            "CompanyCodeCurrency": "moeda",
            "Quantity": "quantidade",
            "BaseUnit": "unidade_medida",
        },
        "chave_negocio": ["id_lancamento"],
    },
}


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 5, Finished, Available, Finished, False)

## 2. Funções utilitárias

Cada função tem uma única responsabilidade — assim o pipeline principal (seção 3) fica curto e fácil de ler.

In [4]:
def ler_bronze(nome_tabela: str):
    """
    Lê uma tabela da camada bronze.

    Não há necessidade de explode/flatten: os dados já estão em formato
    tabular (uma linha por registro). A particularidade é que cada coluna
    tem, literalmente, o prefixo "d.results." no próprio nome.
    """
    return spark.table(f"{LAKEHOUSE_BRONZE}.{SCHEMA_BRONZE}.{nome_tabela}")


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 6, Finished, Available, Finished, False)

In [5]:
def selecionar_colunas(df, colunas: dict):
    """
    Aplica o "de-para" definido em CONFIG_ENTIDADES: mantém apenas as colunas
    relevantes para análise e renomeia cada uma para o nome de negócio.

    Como o nome real da coluna na bronze é "d.results.<campo>" (com pontos
    literais), usamos crase (`` ` ``) ao redor do nome inteiro para o Spark
    tratá-lo como um único identificador — e não como acesso a um campo
    aninhado (d -> results -> campo), que não existe.
    """
    return df.select([
        col(f"`{PREFIXO_COLUNA_BRONZE}{origem}`").alias(destino)
        for origem, destino in colunas.items()
    ])


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 7, Finished, Available, Finished, False)

In [6]:
def analisar_nulos(df, tabela: str):
    """
    Data Quality - Regra 1: completude (valores nulos).

    Para cada coluna, calcula quantos valores são nulos e o percentual
    sobre o total de linhas. Feito em uma única passada pelos dados
    (uma agregação com várias colunas) em vez de um count() por coluna,
    o que evita varrer o DataFrame várias vezes.

    Importante: esta função já recebe o DataFrame COM os nomes de negócio
    (pós-renomeação), então aqui não há mais necessidade de crase.
    """
    total_linhas = df.count()

    # Para cada coluna, gera uma expressão que soma 1 quando o valor é nulo
    # (when/otherwise funciona como um CASE WHEN do SQL)
    agregacoes = [
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]

    # Executa todas as agregações de uma vez só e traz o resultado (1 linha) para o driver
    linha_nulos = df.select(agregacoes).collect()[0].asDict()

    # Monta uma lista de dicionários (uma linha por coluna) para virar
    # um DataFrame de relatório mais adiante
    linhas = []
    for nome_coluna, qtd_nulos in linha_nulos.items():
        pct_nulos = round((qtd_nulos / total_linhas) * 100, 2) if total_linhas > 0 else 0.0
        linhas.append({
            "lakehouse": LAKEHOUSE_SILVER,
            "tabela": tabela,
            "coluna": nome_coluna,
            "qtd_total": total_linhas,
            "qtd_nulos": qtd_nulos,
            "pct_nulos": pct_nulos,
        })
    return linhas


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 8, Finished, Available, Finished, False)

In [7]:
def tratar_duplicidade(df, tabela: str, chave_negocio: list):
    """
    Data Quality - Regra 2: unicidade (duplicidade).

    Remove linhas duplicadas considerando apenas a chave de negócio
    (não a linha inteira) e devolve um resumo de quantas linhas existiam
    antes, depois, e quantas foram removidas.
    """
    qtd_antes = df.count()

    # dropDuplicates com "subset" remove duplicatas olhando só para as colunas
    # da chave de negócio — se houver mais de uma linha com a mesma chave,
    # mantém apenas uma (a escolha entre elas não é determinística por padrão)
    df_dedup = df.dropDuplicates(subset=chave_negocio)
    qtd_depois = df_dedup.count()

    resumo = {
        "lakehouse": LAKEHOUSE_SILVER,
        "tabela": tabela,
        "chave_negocio": ",".join(chave_negocio),
        "qtd_linhas_bronze": qtd_antes,
        "qtd_linhas_silver": qtd_depois,
        "qtd_duplicados_removidos": qtd_antes - qtd_depois,
    }
    return df_dedup, resumo


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 9, Finished, Available, Finished, False)

## 3. Pipeline principal — loop pelas entidades

Percorre cada entidade definida em `CONFIG_ENTIDADES` e aplica, em sequência: leitura da bronze → seleção/renomeação de colunas → checagem de nulos → tratamento de duplicidade → gravação na silver.

In [8]:
# Listas que vão acumular os relatórios de Data Quality de TODAS as entidades,
# para gravar tudo de uma vez só no final (seção 4)
lista_nulos = []
lista_resumo_dq = []

for tabela, cfg in CONFIG_ENTIDADES.items():
    print(f"Processando: {tabela}")

    # 1. Lê a bronze (colunas ainda com o prefixo "d.results.")
    df_bronze = ler_bronze(tabela)

    # 1.1. Se a entidade tiver coluna Language mapeada, filtra apenas registros em PT
    if "Language" in cfg["colunas"]:
        df_bronze = df_bronze.filter(col(f"`{PREFIXO_COLUNA_BRONZE}Language`") == "PT")

    # 1.2. Se a tabela tiver coluna CodeCompany na origem, filtra apenas registros da empresa 1410
    col_codecompany = f"{PREFIXO_COLUNA_BRONZE}CodeCompany"
    if col_codecompany in df_bronze.columns:
        df_bronze = df_bronze.filter(col(f"`{col_codecompany}`") == "1410")

    # 1.3. Regra específica para lancamento_despesas: considerar apenas Ledger = "0L"
    if tabela == "lancamento_despesas":
        df_bronze = df_bronze.filter(col(f"`{PREFIXO_COLUNA_BRONZE}Ledger`") == "0L")

    # 2. Mantém só as colunas relevantes, já sem o prefixo e com nomes de negócio
    df_silver = selecionar_colunas(df_bronze, cfg["colunas"])

    # 2.1. Regras específicas para conta_contabil
    if tabela == "conta_contabil":
        # 2.1.1. Filtrar apenas plano_contas = "YCOA"
        df_silver = df_silver.filter(col("plano_contas") == "YCOA")

        # 2.1.2. Criar coluna "grupo" baseada no início de conta_contabil
        df_silver = df_silver.withColumn(
            "grupo",
            when(col("conta_contabil").startswith("1"), "Cliente")
            .when(col("conta_contabil").startswith("2"), "Fornecedor")
            .when(col("conta_contabil").startswith("4"), "Receita")
            .when(
                (col("conta_contabil").startswith("5")) | (col("conta_contabil").startswith("6")),
                "Custos despesas",
            )
            .otherwise("Outras")
        )

    # 3. Checa nulos ANTES de remover duplicidade, para o relatório refletir
    #    o dado como veio da bronze (após a seleção de colunas)
    lista_nulos.extend(analisar_nulos(df_silver, tabela))

    # 4. Remove duplicidade pela chave de negócio da entidade
    df_silver_dedup, resumo_dq = tratar_duplicidade(df_silver, tabela, cfg["chave_negocio"])
    lista_resumo_dq.append(resumo_dq)

    # 5. Grava a tabela tratada na silver
    #    mode="overwrite" = recarga completa a cada execução (padrão para dimensões pequenas)
    #    option("mergeSchema", "true") permite evolução de schema (novas colunas) sem erro
    df_silver_dedup.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{LAKEHOUSE_SILVER}.{SCHEMA_SILVER}.{tabela}")

    print(f"  -> {resumo_dq['qtd_linhas_silver']} linha(s) gravada(s) em {LAKEHOUSE_SILVER}.{SCHEMA_SILVER}.{tabela}")
    print(f"  -> {resumo_dq['qtd_duplicados_removidos']} duplicado(s) removido(s) (chave: {resumo_dq['chave_negocio']})")

print("\nProcessamento concluído.")


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 10, Finished, Available, Finished, False)

Processando: centro_custo
  -> 40 linha(s) gravada(s) em lh_sap_silver.dbo.centro_custo
  -> 0 duplicado(s) removido(s) (chave: centro_custo)
Processando: centro_lucro
  -> 9 linha(s) gravada(s) em lh_sap_silver.dbo.centro_lucro
  -> 0 duplicado(s) removido(s) (chave: centro_lucro)
Processando: cliente_fornecedor
  -> 116 linha(s) gravada(s) em lh_sap_silver.dbo.cliente_fornecedor
  -> 0 duplicado(s) removido(s) (chave: parceiro_negocio)
Processando: empresa
  -> 1 linha(s) gravada(s) em lh_sap_silver.dbo.empresa
  -> 0 duplicado(s) removido(s) (chave: empresa)
Processando: conta_contabil
  -> 782 linha(s) gravada(s) em lh_sap_silver.dbo.conta_contabil
  -> 0 duplicado(s) removido(s) (chave: plano_contas,conta_contabil)
Processando: segmento
  -> 3 linha(s) gravada(s) em lh_sap_silver.dbo.segmento
  -> 0 duplicado(s) removido(s) (chave: segmento)
Processando: lancamento_despesas
  -> 504 linha(s) gravada(s) em lh_sap_silver.dbo.lancamento_despesas
  -> 0 duplicado(s) removido(s) (chave

## 4. Persistir relatórios de Data Quality

Gravados em modo `append` (diferente das tabelas de negócio, que usam `overwrite`) para manter o histórico de execuções — permitindo acompanhar a evolução da qualidade dos dados ao longo do tempo.

In [9]:
# Schema explícito evita que o Spark tenha que inferir os tipos
# a partir da lista de dicionários (mais rápido e mais seguro)
schema_nulos = StructType([
    StructField("lakehouse", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("coluna", StringType(), False),
    StructField("qtd_total", LongType(), False),
    StructField("qtd_nulos", LongType(), False),
    StructField("pct_nulos", DoubleType(), False),
])

df_dq_nulos = (
    spark.createDataFrame(lista_nulos, schema=schema_nulos)
    # current_timestamp() marca quando essa execução do notebook rodou,
    # permitindo comparar o resultado de uma execução com a anterior
    .withColumn("data_execucao", current_timestamp())
)

df_dq_nulos.write.mode("append").format("delta").saveAsTable(
    f"{LAKEHOUSE_SILVER}.{SCHEMA_SILVER}.ctrl_dq_nulos"
)

display(df_dq_nulos.orderBy("tabela", "coluna"))


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3668e2f0-6cc9-41a5-8dc9-d1918ae5ade1)

In [10]:
schema_resumo_dq = StructType([
    StructField("lakehouse", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("chave_negocio", StringType(), False),
    StructField("qtd_linhas_bronze", LongType(), False),
    StructField("qtd_linhas_silver", LongType(), False),
    StructField("qtd_duplicados_removidos", LongType(), False),
])

df_dq_resumo = (
    spark.createDataFrame(lista_resumo_dq, schema=schema_resumo_dq)
    .withColumn("data_execucao", current_timestamp())
)

df_dq_resumo.write.mode("append").format("delta").saveAsTable(
    f"{LAKEHOUSE_SILVER}.{SCHEMA_SILVER}.ctrl_dq_resumo"
)

display(df_dq_resumo.orderBy("tabela"))


StatementMeta(, 8bac32c5-be0f-4575-9759-b0ec3a443f43, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d59083f-7288-41fc-bb3a-72d3677dbe79)

## 5. Observações sobre o schema bronze atual

- **Nome das colunas:** todas as colunas da bronze têm o prefixo literal `d.results.` no próprio nome (ex: `` `d.results.CompanyCode` ``). Isso é tratado automaticamente pela função `selecionar_colunas`, que monta o nome completo com crase antes de selecionar. Se, em algum momento, a ingestão passar a gravar os dados como struct aninhada de verdade (em vez do nome literal com pontos), essa função precisará voltar a usar `explode` + `.select("r.*")`.
- **lancamento_despesas (ACDOCA):** o schema atual não traz `AccountingDocument`/`AccountingDocumentItem` nem datas de lançamento (`PostingDate`/`DocumentDate`). Usei `ID` como identificador de linha e `LedgerFiscalYear` como referência de ano fiscal. Se análises temporais (evolução mensal etc.) forem necessárias, revise a extração da API para incluir esses campos.
- **centro_custo, centro_lucro, conta_contabil, segmento:** não têm descrição textual embutida — isso vem via navegação `to_Text`, que aparece na bronze só como link `__deferred` (não expandido). Já a `lancamento_despesas` traz textos descritivos prontos (`CostCenterName`, `ProfitCenterName`, `GLAccountName` etc.), que podem servir de referência textual quando necessário.
- A chave de negócio de `lancamento_despesas` (`id_lancamento`) assume que o campo `ID` é único por linha. Se isso não se confirmar na prática (duplicados legítimos com o mesmo ID), amplie a chave combinando mais colunas em `CONFIG_ENTIDADES`.
